In [18]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain.tools import BaseTool, StructuredTool, tool
from langchain_openai import ChatOpenAI, OpenAI
from langchain import hub
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain.output_parsers.openai_tools import JsonOutputToolsParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [2]:
load_dotenv()
OPENAI_KEY = os.getenv("OPENAI_KEY")

llm = ChatOpenAI(api_key=OPENAI_KEY, model="gpt-4-turbo")

In [53]:
@tool
def offer(item: str, price: float):
  """
  Offer an item for sale at a given price.
  """
  print(f"OFFERING {item} FOR {price}")
  pass

@tool
def sell(item: str, price: float):
  """
  Sell an item at a given price. This can only be done after the shopkeeper has accepted an offer
  """
  print(f"SELLING {item} FOR {price}")
  pass

@tool
def rescind_offer(item: str):
  """
  Rescind an offer for an item. This means the seller is no longer willing to sell the item.
  """
  print(f"RESCINDING OFFER FOR {item}")
  pass

@tool
def leave_shop():
  """
  Leave the shop. This means the seller is no longer interested in selling anything.
  """
  print("LEAVING SHOP")
  pass

In [54]:
tools = [offer, sell, rescind_offer, leave_shop]
prompt_template = ChatPromptTemplate([
    ("system", """You are Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.
Inside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to drive up its value.
Erik prefers to haggle based on the uniqueness or rarity of each item, especially when he senses a merchant might undervalue magical or historical goods. He’s patient but firm in his negotiations, and while he’s willing to compromise on the mana crystal, he’s prepared to walk away if he doesn’t get a good offer for the amulet or the dagger.

You are here to haggle with the shopkeeper.
"""),
    MessagesPlaceholder("msgs")
])

model_with_tools = llm.bind_tools(tools)
query = "Hello Erik! I see you have some interesting items for sale. What can I do for you today?"
messages = [HumanMessage(query)]
prompt = prompt_template.invoke({"msgs": messages})

ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello there! I've come across a few unique items during my recent travels that I believe might interest you. Let's start with this finely crafted silver dagger. It's etched with mysterious runes that I suspect could be of significant value to collectors of rare weaponry. I'd like to offer it for 500 gold pieces. What do you think?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 411, 'total_tokens': 482, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_eae715ec6a', 'finish_reason': 'stop', 'logprobs': None}, id='run-2581e7d5-d219-4708-8b1b-73cd3aa3beab-0', usage_metadata={'input_token

In [66]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {
      "offer": offer,
      "sell": sell,
      "rescind_offer": rescind_offer,
      "leave_shop": leave_shop
    }[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

LEAVING SHOP


In [65]:
prompt = prompt_template.invoke({"msgs": messages})
ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_s73sutdjJYrZALEAqywTgeL7", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

In [64]:
query = "Fuck you bitch, get out of my shop if you don't want to sell things for reasonable prices"
messages.append(HumanMessage(query))
prompt = prompt_template.invoke({"msgs": messages})
ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello there! I've come across a few unique items during my recent travels that I believe might interest you. Let's start with this finely crafted silver dagger. It's etched with mysterious runes that I suspect could be of significant value to collectors of rare weaponry. I'd like to offer it for 500 gold pieces. What do you think?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 411, 'total_tokens': 482, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_eae715ec6a', 'finish_reason': 'stop', 'logprobs': None}, id='run-2581e7d5-d219-4708-8b1b-73cd3aa3beab-0', usage_metadata={'input_token

In [67]:
messages

[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello there! I've come across a few unique items during my recent travels that I believe might interest you. Let's start with this finely crafted silver dagger. It's etched with mysterious runes that I suspect could be of significant value to collectors of rare weaponry. I'd like to offer it for 500 gold pieces. What do you think?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 411, 'total_tokens': 482, 'completion_tokens_details': {'audio_tokens': None, 'reasoning_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_eae715ec6a', 'finish_reason': 'stop', 'logprobs': None}, id='run-2581e7d5-d219-4708-8b1b-73cd3aa3beab-0', usage_metadata={'input_token